# Notebook 5: QAOA cho hàm số nguyên (Ví dụ 17–19, Hình 8–10, Bảng 3)

Đi kèm [Phần A, Mục 6](../../docs/A-translation/06-gate-based.md). Notebook chạy hoàn toàn trên trình mô phỏng `AerSimulator`, không cần tài khoản. Ta (1) kiểm tra Hamiltonian ở phương trình (43), (2) dựng mạch của Hình 9, (3) so sánh code **gốc** của Hình 8 và 10 với code **đã sửa**.

In [1]:
import itertools
import numpy as np
from scipy.optimize import minimize
from qiskit.circuit import QuantumCircuit
from qiskit_aer import AerSimulator

def f(x):                                      # hàm mục tiêu của Ví dụ 10 dạng pseudo-Boolean
    return (-9 + 13*x[0] + 14*x[1] + 9*x[2]
            - 18*x[0]*x[1] - 18*x[0]*x[2] - 18*x[1]*x[2] + 36*x[0]*x[1]*x[2])

## 1. Hamiltonian của bài toán (Ví dụ 17, phương trình (43))

$$H_P = -2Z_1 - \tfrac{5}{2}Z_2 - \tfrac{9}{2}Z_1 \otimes Z_2 \otimes Z_3.$$

$H_P$ là diagonal matrix, nên phần tử đường chéo ứng với state $|x\rangle$ phải bằng $f(x)$. Ta dùng quy ước của Qiskit: qubit 0 là bit ít quan trọng nhất, và $x_1$ ứng với qubit 0.

In [2]:
I2, Z = np.eye(2), np.diag([1.0, -1.0])
def op(*ops_by_qubit):                         # ops_by_qubit = [op_q0, op_q1, op_q2]; kron theo thứ tự little-endian
    out = np.array([[1.0]])
    for m in reversed(ops_by_qubit):
        out = np.kron(out, m)
    return out

HP = -2*op(Z, I2, I2) - 2.5*op(I2, Z, I2) - 4.5*op(Z, Z, Z)
diag = np.diag(HP)
ok = True
for idx in range(8):
    x = [(idx >> q) & 1 for q in range(3)]      # x1 = qubit 0, x2 = qubit 1, x3 = qubit 2
    ok &= abs(diag[idx] - f(x)) < 1e-12
print("đường chéo của H_P khớp f(x) trên mọi tổ hợp:", ok)

đường chéo của H_P khớp f(x) trên mọi tổ hợp: True


## 2. Mạch QAOA (Hình 9)

Mỗi lớp gồm $U(H_P, \gamma)$ (hai RZ gate và chuỗi CNOT quanh RZ gate cho term $Z_1 Z_2 Z_3$) rồi mixer $U(H_M, \beta)$ (các RX gate). Mã giống Hình 9 của bài báo.

In [3]:
def build_ansatz(theta):
    num_qubits = 3
    qaoa_depth = len(theta) // 2
    circuit = QuantumCircuit(num_qubits)
    gamma = theta[:qaoa_depth]
    beta = theta[qaoa_depth:]
    for i in range(num_qubits):
        circuit.h(i)
    for i in range(qaoa_depth):
        circuit.rz(-4 * gamma[i], 0)
        circuit.rz(-5 * gamma[i], 1)
        circuit.cx(0, 1)
        circuit.cx(1, 2)
        circuit.rz(-9 * gamma[i], 2)
        circuit.cx(1, 2)
        circuit.cx(0, 1)
        for qbit in range(num_qubits):
            circuit.rx(2 * beta[i], qbit)
    circuit.measure_all()
    return circuit

print(build_ansatz([0.1, 0.1, 0.1, 0.1]).draw(fold=-1))

        ┌───┐┌──────────┐                                      ┌─────────┐┌──────────┐                                      ┌─────────┐ ░ ┌─┐      
   q_0: ┤ H ├┤ Rz(-0.4) ├──■─────────────────────────────■─────┤ Rx(0.2) ├┤ Rz(-0.4) ├──■─────────────────────────────■─────┤ Rx(0.2) ├─░─┤M├──────
        ├───┤├──────────┤┌─┴─┐                         ┌─┴─┐   ├─────────┤├──────────┤┌─┴─┐                         ┌─┴─┐   ├─────────┤ ░ └╥┘┌─┐   
   q_1: ┤ H ├┤ Rz(-0.5) ├┤ X ├──■────────────────■─────┤ X ├───┤ Rx(0.2) ├┤ Rz(-0.5) ├┤ X ├──■────────────────■─────┤ X ├───┤ Rx(0.2) ├─░──╫─┤M├───
        ├───┤└──────────┘└───┘┌─┴─┐┌──────────┐┌─┴─┐┌──┴───┴──┐└─────────┘└──────────┘└───┘┌─┴─┐┌──────────┐┌─┴─┐┌──┴───┴──┐└─────────┘ ░  ║ └╥┘┌─┐
   q_2: ┤ H ├─────────────────┤ X ├┤ Rz(-0.9) ├┤ X ├┤ Rx(0.2) ├────────────────────────────┤ X ├┤ Rz(-0.9) ├┤ X ├┤ Rx(0.2) ├────────────░──╫──╫─┤M├
        └───┘                 └───┘└──────────┘└───┘└─────────┘                            └───┘└──────────┘└───

## 3. Lỗi trong code gốc (Hình 8 và 10)

**(a) `nshots` bị bỏ qua.** Tên tham số đúng của Aer là `shots`.

In [4]:
sim = AerSimulator()
c = build_ansatz([0.1, 0.1, 0.1, 0.1])
print("run(..., nshots=1000) -> tổng số mẫu:", sum(sim.run(c, nshots=1000).result().get_counts().values()))
print("run(..., shots=1000)  -> tổng số mẫu:", sum(sim.run(c, shots=1000).result().get_counts().values()))

run(..., nshots=1000) -> tổng số mẫu: 1024
run(..., shots=1000)  -> tổng số mẫu: 1000


**(b) `value = ...` thay vì `value += ...`.** Dưới đây là hai hàm mục tiêu cổ điển: bản gốc (giữ nguyên lỗi) và bản đã sửa. Ta dùng cùng `seed_simulator` cho cả hai để kết quả lặp lại được.

In [5]:
SEED = 7

def classical_obj_fn_original(theta, num_executions=1000):      # Hình 10 gốc: value = ... (ghi đè)
    counts = sim.run(build_ansatz(theta), shots=num_executions, seed_simulator=SEED).result().get_counts()
    value = 0
    sampled = 0
    for solution, occurrences in counts.items():
        x = [int(v) for v in solution[::-1]]
        value = f(x) * occurrences                               # BUG: không cộng dồn
        sampled += occurrences
    return value / sampled

def classical_obj_fn_fixed(theta, num_executions=1000):         # đã sửa: value += ...
    counts = sim.run(build_ansatz(theta), shots=num_executions, seed_simulator=SEED).result().get_counts()
    value = 0
    sampled = 0
    for solution, occurrences in counts.items():
        x = [int(v) for v in solution[::-1]]
        value += f(x) * occurrences
        sampled += occurrences
    return value / sampled

## 4. Chạy COBYLA với từng hàm mục tiêu và so sánh

Sau mỗi lần tối ưu, ta đánh giá lại tham số tìm được bằng 20000 shots để ước lượng giá trị trung bình thật của $f$ và measurement probability được solution tối ưu $(0,0,0)$.

In [6]:
def assess(theta, shots=20000, seed=5):
    counts = sim.run(build_ansatz(theta), shots=shots, seed_simulator=seed).result().get_counts()
    mean = sum(f([int(v) for v in k[::-1]]) * n for k, n in counts.items()) / shots
    return mean, counts.get("000", 0) / shots

results = {}
for name, fn in (("gốc (value = ...)", classical_obj_fn_original), ("đã sửa (value += ...)", classical_obj_fn_fixed)):
    res = minimize(fn, [0.1, 0.1, 0.1, 0.1], method="COBYLA")
    mean, p000 = assess(res.x)
    results[name] = res.x
    print(f"{name:24s} tham số={np.round(res.x, 3)}  giá trị trung bình thật của f = {mean:6.2f}   P(000) = {p000:.3f}")
print("tham chiếu: distribution đều cho trung bình 0.00 và P(000) = 0.125; giá trị nhỏ nhất của f là -9")

gốc (value = ...)        tham số=[0.076 1.141 0.12  0.263]  giá trị trung bình thật của f =  -1.96   P(000) = 0.164
đã sửa (value += ...)    tham số=[1.176 0.09  0.16  1.151]  giá trị trung bình thật của f =  -4.49   P(000) = 0.304
tham chiếu: distribution đều cho trung bình 0.00 và P(000) = 0.125; giá trị nhỏ nhất của f là -9


Hàm mục tiêu **đã sửa** cho giá trị trung bình thấp hơn rõ rệt và measurement probability được solution tối ưu cao hơn. Lưu ý: bản gốc trả về giá trị phụ thuộc vào thứ tự khóa trong từ điển kết quả, nên nó không phải là hàm tốt cho COBYLA. Kết quả chính xác thay đổi theo hạt giống và điểm khởi đầu, nên hãy xem đây là minh họa, không phải measurement chặt chẽ.

## 5. Kết quả sampling với tham số đã tối ưu (so với Bảng 3)

Khóa của Qiskit có dạng `q2 q1 q0`, nên khi đọc $(x_1, x_2, x_3)$ ta đảo chuỗi.

In [7]:
theta = results["đã sửa (value += ...)"]
counts = sim.run(build_ansatz(theta), shots=1000, seed_simulator=11).result().get_counts()
print("Khóa Qiskit | (x1,x2,x3) |  f(x) | Occurrences")
for key, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    x = tuple(int(v) for v in key[::-1])
    print(f"    {key}     |  {x}  | {f(x):5d} | {n:5d}")
print("tổng:", sum(counts.values()))

Khóa Qiskit | (x1,x2,x3) |  f(x) | Occurrences
    000     |  (0, 0, 0)  |    -9 |   312
    110     |  (0, 1, 1)  |    -4 |   285
    101     |  (1, 0, 1)  |    -5 |   166
    011     |  (1, 1, 0)  |     0 |   147
    010     |  (0, 1, 0)  |     5 |    41
    100     |  (0, 0, 1)  |     0 |    27
    001     |  (1, 0, 0)  |     4 |    16
    111     |  (1, 1, 1)  |     9 |     6
tổng: 1000
